In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch


CACHE_DIR = "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/sequence_cache_uid_disjoint/v4_per_uid_timestep"

In [2]:
import importlib
import gtan_augment_ieee
importlib.reload(gtan_augment_ieee)

from gtan_augment_ieee import augment_with_gtan_embeddings_from_cache
from gtan_augment_ieee import augment_with_gtan_embeddings_trainval

In [3]:
df = pd.read_parquet(
    "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/parquet_exported_files/X_train_copy4.parquet"
)

In [10]:
df

,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,DT_M_total,DT_W_total,DT_D_total,uid_FE,uid_v2_FE,D1n,D2n,D3n,D5n,D9n
TransactionID,,,,,,,,,,,,,,,,,,,,,
2987000.0,86400.0,68.500000,4,12926.0,-1.0,50.0,1,42.0,1,215.0,...,137321,12093,5122,1.0,NaN,-13.0,2.0,-12.0,2.0,2.0
2987001.0,86401.0,29.000000,4,1755.0,304.0,50.0,2,2.0,1,225.0,...,137321,12093,5122,1.0,1.0,1.0,2.0,2.0,2.0,2.0
2987002.0,86469.0,59.000000,4,3663.0,390.0,50.0,3,66.0,2,230.0,...,137321,12093,5122,4.0,2.0,1.0,2.0,2.0,2.0,2.0
2987003.0,86499.0,50.000000,4,17132.0,467.0,50.0,2,17.0,2,376.0,...,137321,12093,5122,84.0,81.0,-111.0,-111.0,1.0,1.0,2.0
2987004.0,86506.0,50.000000,1,3497.0,414.0,50.0,2,2.0,1,320.0,...,137321,12093,5122,1.0,NaN,1.0,2.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3577535.0,15811047.0,49.000000,4,5550.0,-1.0,50.0,3,126.0,2,172.0,...,89326,10288,2754,2.0,2.0,153.0,153.0,152.0,183.0,183.0
3577536.0,15811049.0,39.500000,4,9444.0,125.0,50.0,2,124.0,2,104.0,...,89326,10288,2754,1.0,NaN,182.0,183.0,183.0,183.0,183.0
3577537.0,15811079.0,30.950001,4,11037.0,495.0,50.0,2,124.0,2,131.0,...,89326,10288,2754,9.0,6.0,182.0,183.0,183.0,183.0,183.0


In [7]:
df.index

Index([2987000.0, 2987001.0, 2987002.0, 2987003.0, 2987004.0, 2987005.0,
       2987006.0, 2987007.0, 2987008.0, 2987009.0,
       ...
       3577530.0, 3577531.0, 3577532.0, 3577533.0, 3577534.0, 3577535.0,
       3577536.0, 3577537.0, 3577538.0, 3577539.0],
      dtype='float64', name='TransactionID', length=590540)

In [ ]:
# pick GPU on Kaggle; falls back to CPU if accelerator is off
device = "cuda" if torch.cuda.is_available() else "cpu"

X_train = pd.read_parquet(
    "/kaggle/input/<your-dataset>/X_train_copy4.parquet"  # or wherever you upload it
)
y_train = pd.read_parquet("/kaggle/input/<your-dataset>/y_train.parquet")["isFraud"]

res_cache = augment_with_gtan_embeddings_from_cache(
    df=X_train,                 # has uid / card1_addr1 / ... / DeviceInfo / TransactionDT
    cache_dir=CACHE_DIR,
    # ---- GTAN-size capacity (width = 64*4 = 256) ----
    hidden_dim=32,
    heads=4,
    n_layers=2,
    dropout=0.2,
    # ---- training ----
    n_epochs=30,
    lr=3e-4,
    label_rate=0.5,             # keep at 0.5
    # ---- Kaggle GPU + memory-safe minibatching ----
    device=device,              # <-- overrides the mps default; REQUIRED
    batch_size=4096,            # full-graph -> OOM risk at width 256; minibatch instead
    num_neighbors=[-1, -1],     # full neighbours per hop (faithful); use [15, 10] if still tight
    verbose=True,
    prefix="gtan_cache_"
)
print(res_cache["val_auc"], res_cache["val_ap"])

In [ ]:
X_train = pd.read_parquet(".../X_train_copy4.parquet")
y_train = pd.read_parquet(".../y_train.parquet")["isFraud"]

res_df = augment_with_gtan_embeddings_trainval(
    X_train, y_train,
    hidden_dim=64, heads=4, n_layers=2, dropout=0.2,
    n_epochs=30, lr=3e-4, label_rate=0.5,
    device=device,
    batch_size=4096, num_neighbors=[-1, -1],   # minibatch — safe at width 256
    prefix="gtan_df_",        # <- distinct prefix so it won't collide with the cache run
    verbose=True,
)

In [ ]:
from pathlib import Path
import json
OUT = Path(".../gtan_outputs"); OUT.mkdir(parents=True, exist_ok=True)

def save_result(res, tag):
    res["emb_train"].to_parquet(OUT / f"emb_train_{tag}.parquet")
    res["emb_val"].to_parquet(OUT / f"emb_val_{tag}.parquet")
    json.dump({"val_auc": res["val_auc"], "val_ap": res["val_ap"],
               "n_train": res["n_train"], "n_val": res["n_val"]},
              open(OUT / f"metrics_{tag}.json", "w"), indent=2)
    torch.save(res["model"].state_dict(), OUT / f"model_{tag}.pt")   # optional

save_result(res_df,    "df")
save_result(res_cache, "cache")